# Modeling: transit ridership forecasting

This notebook trains three models (Ridge Regression, Random Forest, XGBoost)
on the engineered feature set and compares their performance using a temporal
train/test split and TimeSeriesSplit cross-validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.data_loader import load_or_fetch_ridership, preprocess_ridership, engineer_features
from src.model import prepare_model_data, train_models, NUMERICAL_FEATURES, TARGET

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Prepare data

In [ ]:
df_raw = load_or_fetch_ridership('../data', limit=100000)
df = preprocess_ridership(df_raw)
df = engineer_features(df)

X, y, _, feature_names = prepare_model_data(df)
print(f'Feature matrix: {X.shape}')
print(f'Target vector: {y.shape}')
print(f'Features: {feature_names}')

## 2. Temporal train/test split

For time-series data we must respect chronological order. The last 12 months
serve as the hold-out test set, simulating a real forecasting scenario.

In [ ]:
n_test = min(12, len(X) // 5)
if n_test < 2:
    n_test = max(2, int(len(X) * 0.2))

X_train, X_test = X.iloc[:-n_test], X.iloc[-n_test:]
y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'Train period: index {X_train.index[0]} to {X_train.index[-1]}')
print(f'Test period:  index {X_test.index[0]} to {X_test.index[-1]}')

In [ ]:
# Scale features for Ridge regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Scaling applied. Feature means (train):', np.round(X_train_scaled.mean(axis=0), 4))

## 3. Train Ridge regression

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = np.maximum(ridge.predict(X_test_scaled), 0)

ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print(f'Ridge Regression results:')
print(f'  MAE:  {ridge_mae:,.2f}')
print(f'  MSE:  {ridge_mse:,.2f}')
print(f'  R2:   {ridge_r2:.4f}')

# Ridge coefficients
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': ridge.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print('\nRidge coefficients (sorted by magnitude):')
coef_df

## 4. Train Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_split=5,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred_rf = np.maximum(rf.predict(X_test), 0)

rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)

print(f'Random Forest results:')
print(f'  MAE:  {rf_mae:,.2f}')
print(f'  MSE:  {rf_mse:,.2f}')
print(f'  R2:   {rf_r2:.4f}')

## 5. Train XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, n_jobs=-1
)
xgb.fit(X_train, y_train)
y_pred_xgb = np.maximum(xgb.predict(X_test), 0)

xgb_mae = mean_absolute_error(y_test, y_pred_xgb)
xgb_mse = mean_squared_error(y_test, y_pred_xgb)
xgb_r2 = r2_score(y_test, y_pred_xgb)

print(f'XGBoost results:')
print(f'  MAE:  {xgb_mae:,.2f}')
print(f'  MSE:  {xgb_mse:,.2f}')
print(f'  R2:   {xgb_r2:.4f}')

## 6. TimeSeriesSplit cross-validation

TimeSeriesSplit ensures that each fold uses only past data for training
and future data for validation, preventing look-ahead bias.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

models_cv = {
    'Ridge': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=10, min_samples_split=5,
        random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1
    ),
}

cv_results = {}
for name, model in models_cv.items():
    # Use scaled data for Ridge, raw for tree models
    X_cv = X_train_scaled if name == 'Ridge' else X_train
    scores = cross_val_score(model, X_cv, y_train, cv=tscv,
                             scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:20s}  CV R2: {scores.mean():.4f} (+/- {scores.std():.4f})  '
          f'folds: {np.round(scores, 4)}')

In [ ]:
# Visualize cross-validation results
fig, ax = plt.subplots(figsize=(8, 4))
positions = range(len(cv_results))
names = list(cv_results.keys())
bp = ax.boxplot([cv_results[n] for n in names], positions=positions,
                patch_artist=True, widths=0.5)

colors = ['#90CAF9', '#4CAF50', '#FF9800']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xticks(positions)
ax.set_xticklabels(names)
ax.set_ylabel('R2 score')
ax.set_title('TimeSeriesSplit CV comparison (5 folds)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 7. Model comparison summary

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Ridge Regression', 'Random Forest', 'XGBoost'],
    'MAE': [ridge_mae, rf_mae, xgb_mae],
    'MSE': [ridge_mse, rf_mse, xgb_mse],
    'R2': [ridge_r2, rf_r2, xgb_r2],
}).set_index('Model')

print('Hold-out test set results:')
comparison.style.format({'MAE': '{:,.2f}', 'MSE': '{:,.2f}', 'R2': '{:.4f}'}).background_gradient(
    subset=['R2'], cmap='Greens'
)

In [ ]:
# Side-by-side bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

model_names = comparison.index.tolist()
colors = ['#90CAF9', '#4CAF50', '#FF9800']

for ax, metric in zip(axes, ['MAE', 'MSE', 'R2']):
    values = comparison[metric].values
    ax.bar(model_names, values, color=colors)
    ax.set_title(metric)
    ax.set_ylabel(metric)
    for i, v in enumerate(values):
        fmt = f'{v:,.0f}' if metric != 'R2' else f'{v:.3f}'
        ax.text(i, v, fmt, ha='center', va='bottom', fontsize=9)

plt.suptitle('Model comparison on hold-out test set', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Use the pipeline from model.py

As a validation step, we also run the full `train_models()` pipeline
from `src/model.py` and confirm the results are consistent.

In [ ]:
trained_models, results, scaler_pipe, X_test_pipe, y_test_pipe = train_models(X, y)

print('Pipeline results:')
for name, metrics in results.items():
    print(f'  {name:20s}  MAE={metrics["MAE"]:,.2f}  '
          f'RMSE={metrics["RMSE"]:,.2f}  R2={metrics["R2"]:.4f}')

## Summary

All three models were trained on temporally split data and evaluated with
TimeSeriesSplit cross-validation. Key findings:

- **XGBoost** and **Random Forest** tend to outperform Ridge on this dataset,
  as they can capture nonlinear interactions between lag and seasonal features
- **Ridge** provides a strong linear baseline and interpretable coefficients
- **TimeSeriesSplit CV** confirms that performance is stable across different
  training windows

Next: detailed evaluation, residual analysis, and feature importance in
`04_evaluation.ipynb`.